# Severity: transformer K-fold on the original CNN dataset

Five `StratifiedGroupKFold` folds, `shuffle=True`, seed **42**, and original video/image identifiers match the uploaded CNN K-fold notebooks. This notebook trains **transformers only** and writes to its own output folder.

Use **Kaggle GPU T4 x2** with Internet enabled. Run the numbered sections in order. Four models are configured by default: ViT-B/16, DeiT-Small, Swin-Tiny, and Swin-Small. These are an editable candidate shortlist, **not a claimed performance ranking**: the supplied transformer notebook copies contain no saved scores. If a prior transformer `summary.csv` is supplied in Section 2, Section 7 automatically chooses the top four by **validation macro-F1** for this task.

Training keeps the supplied transformer preprocessing, full-backbone fine-tuning, feature L2 normalization, two learning rates, and task-specific loss. The fast profile is at most **20 epochs**, patience **4**. Twenty model/fold jobs run at most two at a time, one per GPU. Each interrupted fold resumes from its last fully saved epoch; completed folds are skipped.

The held-out fold selects the checkpoint by validation loss and provides the reported fold metrics, matching the existing CNN project's validation-fold selection approach. These are **validation CV scores**, rather than an additional independent test evaluation. No CNN rerun or modification is performed.

Original source links copied from the uploaded CNN notebook:

- [Image ZIP](https://drive.google.com/file/d/117lqrb69UR41ZeQXtTX895dE3sF8iTmo/view?usp=sharing)
- [Metadata editor/view link](https://docs.google.com/spreadsheets/d/1YSX2UP_paaqiCQ9W1mzVKub7GvJDL7i7/edit?usp=sharing)


## 1. Dependencies
Keep Kaggle's installed PyTorch, torchvision, and CUDA pair. The notebook installs the same `timm` version as the supplied transformer notebooks. There is no GitHub clone and no modification of the CNN project.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'timm==1.0.22', 'gdown==5.2.0', 'openpyxl'], check=True)


## 2. Configuration
`MODEL_NAMES` is the candidate list when `PRIOR_RESULTS_CSV=None`. To select the actual top four from previous results, set `PRIOR_RESULTS_CSV` to this task's previous transformer summary CSV, with columns `model` and `val_macro_f1` (or `validation_macro_f1`). Test metrics are not used to shortlist models.

The original dataset URLs and label checks below are already filled in. `LOCAL_FILES` can point to Kaggle input files instead of downloading. `CNN_FOLDS_CSV` is optional: providing the original CNN `fold_assignments.csv` guarantees reuse of its exact assignments. Otherwise the same seed-42 grouped splitter is used, and the generated assignments are saved for every transformer.

For a new Kaggle session, upload your exported transformer output ZIP to Drive and paste its view link in `RESUME_OUTPUTS_ZIP_URL`. Use the same `RUN_NAME`, model settings, and notebook. Do not put a CNN output ZIP or old single-split transformer checkpoints there.


In [ ]:
from pathlib import Path
import json, os

ROOT = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
TASK = 'severity'
RUN_NAME = 'transformer_kfold_' + TASK + '_original_ids_fast_v1'
TASK_ROOT = ROOT / 'outputs' / RUN_NAME / TASK
DATA_ROOT = ROOT / ('transformer_kfold_data_' + TASK)
PROJECT_DIR = ROOT / ('transformer_kfold_project_' + TASK)

RUN_TRAINING = True
N_SPLITS = 5
SEED = 42
TOP_N = 4
MODEL_NAMES = ['ViT-B/16', 'DeiT-Small', 'Swin-Tiny', 'Swin-Small']
PRIOR_RESULTS_CSV = None       # Example: '/kaggle/input/my-results/summary.csv'
CNN_FOLDS_CSV = None           # Optional original CNN fold_assignments.csv
LOCAL_FILES = {'zip': None, 'metadata': None}
RESUME_OUTPUTS_ZIP_URL = ''    # Paste the Drive view link of THIS notebook's output ZIP
RESUME_LOCAL_ZIP = None        # Alternative: local path to that exported ZIP
VALIDATE_IMAGES = True

TRAINING = dict(
    epochs=20, patience=4, batch_size=32, workers=2,
    backbone_lr=1e-5, head_lr=1e-4, weight_decay=0.01,
    pretrained=True, keep_last_for_completed=False,
)
# Finished folds keep best.pth; their large optimizer/resume states are removed
# to save disk. Unfinished folds always retain last.pth for resume.

SOURCE = {'zip_id': '117lqrb69UR41ZeQXtTX895dE3sF8iTmo', 'metadata_id': '1YSX2UP_paaqiCQ9W1mzVKub7GvJDL7i7', 'zip_url': 'https://drive.google.com/file/d/117lqrb69UR41ZeQXtTX895dE3sF8iTmo/view?usp=sharing', 'metadata_url': 'https://docs.google.com/spreadsheets/d/1YSX2UP_paaqiCQ9W1mzVKub7GvJDL7i7/edit?usp=sharing', 'zip_filename': 'Dataset_Severity_V2.zip', 'metadata_filename': 'Dataset_Severity_V2.xlsx', 'metadata_kind': 'xlsx', 'classes': ['low nuclear density', 'dense', 'mature', 'brunescent'], 'rows': 2541, 'groups': 645, 'counts': {0: 795, 1: 1047, 2: 614, 3: 85}, 'expected_hash': '1bcc1c61c024845835cabc7ad020f6d1128a2a3ff929e03bf0d8fd53ee4b5e50'}
print('Task:', TASK)
print('ZIP:', SOURCE['zip_url'])
print('Metadata:', SOURCE['metadata_url'])
print('Outputs:', TASK_ROOT)
print('Budget:', TRAINING)


## 3. GPU and environment
Two GPUs train independent model/fold jobs. Each model fits on one GPU, so the batch size is per model, not divided between devices. A single-GPU session also works, sequentially.


In [ ]:
import torch, torchvision, timm, sklearn, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__, 'torchvision:', torchvision.__version__)
print('timm:', timm.__version__, 'scikit-learn:', sklearn.__version__)
print('CUDA available:', torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print('GPU', i, torch.cuda.get_device_name(i),
          round(torch.cuda.get_device_properties(i).total_memory / 1024**3, 2), 'GB')
if RUN_TRAINING and not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator, preferably T4 x2.')
if RUN_TRAINING and torch.cuda.device_count() < 2:
    print('One GPU detected: jobs will run sequentially.')


## 4. Create the standalone transformer project
This cell embeds the complete data, model, training, resume, metrics, and worker-launch code. Run it unchanged. Training workers start in fresh Python processes with their GPU assigned before CUDA initialization.


In [ ]:
import importlib.util, sys, shutil
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
TASK_ROOT.mkdir(parents=True, exist_ok=True)
ENGINE_SOURCE = '"""Standalone transformer K-fold engine embedded in both delivered notebooks."""\nfrom pathlib import Path\nfrom collections import defaultdict\nimport argparse\nimport gc\nimport hashlib\nimport json\nimport os\nimport platform\nimport random\nimport signal\nimport subprocess\nimport sys\nimport time\nimport zipfile\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport sklearn\nfrom sklearn.model_selection import StratifiedGroupKFold\nfrom sklearn.metrics import (\n    accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support,\n    roc_auc_score,\n)\nimport torch\nfrom torch import nn\nfrom torch.utils.data import Dataset, DataLoader, WeightedRandomSampler\nfrom torchvision import transforms\nimport timm\n\nREGISTRY = {\n    \'ViT-B/16\': \'vit_base_patch16_224\',\n    \'ViT-B/32\': \'vit_base_patch32_224\',\n    \'DeiT-Tiny\': \'deit_tiny_patch16_224\',\n    \'DeiT-Small\': \'deit_small_patch16_224\',\n    \'DeiT-Base\': \'deit_base_patch16_224\',\n    \'DeiT-Small-Distilled\': \'deit_small_distilled_patch16_224\',\n    \'Swin-Tiny\': \'swin_tiny_patch4_window7_224\',\n    \'Swin-Small\': \'swin_small_patch4_window7_224\',\n    \'Swin-Base\': \'swin_base_patch4_window7_224\',\n}\nENGINE_VERSION = \'original_ids_grouped_kfold_v1\'\n\n\ndef require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef json_safe(value):\n    if isinstance(value, dict):\n        return {str(k): json_safe(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [json_safe(v) for v in value]\n    if isinstance(value, np.integer):\n        return int(value)\n    if isinstance(value, (np.floating, float)):\n        return float(value) if np.isfinite(value) else None\n    if isinstance(value, Path):\n        return str(value)\n    return value\n\n\ndef save_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + \'.tmp\')\n    temp.write_text(json.dumps(json_safe(value), indent=2, sort_keys=True,\n                               allow_nan=False), encoding=\'utf-8\')\n    temp.replace(path)\n\n\ndef file_hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef table_hash(df, columns):\n    stable = df[columns].sort_values([\'video_id\', \'file_name\']).reset_index(drop=True)\n    return hashlib.sha256(stable.to_csv(index=False, lineterminator=\'\\n\').encode()).hexdigest()\n\n\ndef model_slug(name):\n    return name.replace(\'/\', \'_\')\n\n\ndef canonical_metadata(path, source):\n    """Match the original-ID CNN notebooks, including PD exact-record deduplication."""\n    if source[\'metadata_kind\'] == \'xlsx\':\n        raw = pd.read_excel(path, sheet_name=\'Dataset_Severity_V2\')\n    else:\n        raw = pd.read_csv(path)\n    raw.columns = raw.columns.astype(str).str.strip()\n    needed = [\'filename\', \'videoname\'] + source[\'classes\']\n    require(set(needed).issubset(raw.columns), f\'Missing original metadata columns: {needed}\')\n    require(not raw[needed].isna().any().any(), \'Missing filename/video/label value\')\n    onehot = raw[source[\'classes\']]\n    require(onehot.isin([0, 1]).all().all() and onehot.sum(axis=1).eq(1).all(),\n            \'Every row must contain exactly one valid one-hot class\')\n    df = pd.DataFrame({\n        \'file_name\': raw.filename.astype(str).str.strip(),\n        \'video_id\': raw.videoname.astype(str).str.strip(),\n        \'label\': onehot.to_numpy().argmax(axis=1).astype(int),\n    })\n    require(df.file_name.ne(\'\').all() and df.video_id.ne(\'\').all(), \'Empty identifiers\')\n    require(df.groupby([\'video_id\', \'file_name\']).label.nunique().max() == 1,\n            \'Conflicting labels for one file; do not silently discard a conflict\')\n    before = len(df)\n    # Preserve original row order, as in the CNN notebooks.\n    df = df.drop_duplicates([\'video_id\', \'file_name\'], keep=\'first\').reset_index(drop=True)\n    require(not df.file_name.duplicated().any(), \'A filename belongs to multiple video records\')\n    require(len(df) == source[\'rows\'] and df.video_id.nunique() == source[\'groups\'],\n            f"Expected {source[\'rows\']} records / {source[\'groups\']} video groups; "\n            f\'found {len(df)} / {df.video_id.nunique()}\')\n    require(df.label.value_counts().sort_index().to_dict() ==\n            {int(k): int(v) for k, v in source[\'counts\'].items()}, \'Class counts differ from CNN data\')\n    require(table_hash(df, [\'file_name\', \'video_id\', \'label\']) == source[\'expected_hash\'],\n            \'Metadata differs from the original CNN label records. Check source links, not mapped files.\')\n    if len(source[\'classes\']) == 4:\n        require(df.groupby(\'video_id\').label.nunique().eq(1).all(), \'Mixed severity labels\')\n    df.insert(0, \'row_id\', np.arange(len(df), dtype=int))\n    print(f\'Original metadata verified: {len(df)} frames, {df.video_id.nunique()} groups; \'\n          f\'{before - len(df)} duplicate record(s) removed.\', flush=True)\n    return df\n\n\ndef fetch_source(settings, kind):\n    import gdown\n    import requests\n    source = settings[\'source\']\n    override = settings[\'local_files\'].get(kind)\n    if override:\n        path = Path(override)\n        require(path.is_file(), f\'Local override does not exist: {path}\')\n        return path\n    # Reuse the CNN downloads read-only if they are present in this Kaggle session.\n    previous = Path(settings[\'root\']) / \'downloads\' / source[kind + \'_filename\']\n    if previous.is_file() and (kind != \'zip\' or zipfile.is_zipfile(previous)):\n        print(\'Reusing existing original download:\', previous, flush=True)\n        return previous\n    directory = Path(settings[\'data_root\']) / \'downloads\'\n    directory.mkdir(parents=True, exist_ok=True)\n    path = directory / source[kind + \'_filename\']\n    if path.is_file() and path.stat().st_size and (kind != \'zip\' or zipfile.is_zipfile(path)):\n        return path\n    temp = path.with_suffix(path.suffix + \'.part\')\n    if temp.exists():\n        temp.unlink()\n    if kind == \'metadata\' and source[\'metadata_kind\'] == \'xlsx\':\n        response = requests.get(\n            f"https://docs.google.com/spreadsheets/d/{source[\'metadata_id\']}/export?format=xlsx",\n            timeout=180,\n        )\n        response.raise_for_status()\n        temp.write_bytes(response.content)\n    else:\n        result = gdown.download(id=source[kind + \'_id\'], output=str(temp), quiet=False)\n        require(result is not None, \'Drive download failed; check sharing or set LOCAL_FILES\')\n    require(temp.is_file() and temp.stat().st_size > 0, \'Empty download\')\n    if kind == \'zip\' or source[\'metadata_kind\'] == \'xlsx\':\n        require(zipfile.is_zipfile(temp), f\'Expected ZIP/XLSX, received another file: {path}\')\n    else:\n        require(b\'<html\' not in temp.read_bytes()[:500].lower(), \'Received HTML instead of metadata CSV\')\n    temp.replace(path)\n    return path\n\n\ndef validate_zip_members(archive, root):\n    root = Path(root).resolve()\n    seen = set()\n    for info in archive.infolist():\n        target = (root / info.filename).resolve()\n        require(target.is_relative_to(root), f\'ZIP path escapes extraction directory: {info.filename}\')\n        require((info.external_attr >> 16) & 0o170000 != 0o120000, \'ZIP symlinks are not supported\')\n        if not info.is_dir():\n            require(target not in seen, f\'Duplicate ZIP file member: {info.filename}\')\n            seen.add(target)\n\n\ndef extract_images(path, data_root):\n    digest = file_hash(path)\n    root = Path(data_root) / (\'images_\' + digest[:16])\n    marker = root / \'extraction_complete.json\'\n    if marker.exists():\n        require(json.loads(marker.read_text())[\'sha256\'] == digest, \'Archive checksum mismatch\')\n        return root, digest\n    root.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(path) as archive:\n        validate_zip_members(archive, root)\n        archive.extractall(root)\n    save_json(marker, {\'sha256\': digest})\n    return root, digest\n\n\ndef attach_images(df, root, validate=True):\n    index = defaultdict(list)\n    extensions = {\'.jpg\', \'.jpeg\', \'.png\', \'.bmp\', \'.tif\', \'.tiff\', \'.webp\'}\n    for path in sorted(Path(root).rglob(\'*\')):\n        if path.is_file() and path.suffix.lower() in extensions:\n            index[(path.parent.name.strip(), path.name.strip())].append(path)\n    paths = []\n    for row in df.itertuples(index=False):\n        matches = index[(row.video_id, row.file_name)]\n        require(len(matches) == 1, f\'Missing/ambiguous original image: {row.video_id}/{row.file_name}\')\n        path = matches[0]\n        if validate:\n            with Image.open(path) as image:\n                image.convert(\'RGB\').load()\n        paths.append(str(path.resolve()))\n    out = df.copy()\n    out[\'image_path\'] = paths\n    print(\'Original images resolved:\', len(out), flush=True)\n    return out\n\n\ndef validate_folds(df, n_splits):\n    require(set(df.fold.unique()) == set(range(n_splits)), \'Incomplete/invalid fold numbers\')\n    require(df.groupby(\'video_id\').fold.nunique().eq(1).all(), \'A video crosses folds\')\n    classes = set(df.label.unique())\n    for fold in range(n_splits):\n        train, valid = df[df.fold != fold], df[df.fold == fold]\n        require(not (set(train.video_id) & set(valid.video_id)), f\'Group overlap in fold {fold}\')\n        require(set(train.label) == classes and set(valid.label) == classes,\n                f\'Class missing from train/validation fold {fold}\')\n\n\ndef assign_folds(df, n_splits=5, seed=42, existing_csv=None):\n    out = df.copy()\n    if existing_csv:\n        saved = pd.read_csv(existing_csv)\n        saved = saved.rename(columns={\'filename\': \'file_name\', \'videoname\': \'video_id\',\n                                      \'true_label\': \'label\'})\n        needed = [\'file_name\', \'video_id\', \'label\', \'fold\']\n        require(set(needed).issubset(saved), \'Fold CSV lacks original identifiers, labels or fold\')\n        require(not saved.duplicated([\'file_name\', \'video_id\']).any(), \'Duplicate fold records\')\n        require(len(saved) == len(df), \'Fold manifest belongs to a different dataset\')\n        merged = out.merge(saved[needed], on=[\'file_name\', \'video_id\'], how=\'left\',\n                           validate=\'one_to_one\', suffixes=(\'\', \'_saved\'), sort=False)\n        require(not merged.fold.isna().any() and merged.label.eq(merged.label_saved).all(),\n                \'Fold CSV differs from the original dataset labels or identifiers\')\n        require(merged.fold.isin(range(n_splits)).all(), \'Fold CSV contains invalid fold numbers\')\n        out[\'fold\'] = merged.fold.astype(int).to_numpy()\n    else:\n        out[\'fold\'] = -1\n        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)\n        for fold, (_, indices) in enumerate(splitter.split(out, out.label, out.video_id)):\n            out.loc[out.index[indices], \'fold\'] = fold\n    validate_folds(out, n_splits)\n    return out\n\n\ndef prepare_task(settings):\n    task_root = Path(settings[\'task_root\'])\n    task_root.mkdir(parents=True, exist_ok=True)\n    metadata_path = fetch_source(settings, \'metadata\')\n    zip_path = fetch_source(settings, \'zip\')\n    df = canonical_metadata(metadata_path, settings[\'source\'])\n    extracted, archive_hash = extract_images(zip_path, settings[\'data_root\'])\n    df = attach_images(df, extracted, settings[\'validate_images\'])\n    saved = task_root / \'fold_assignments.csv\'\n    supplied = settings.get(\'cnn_folds_csv\')\n    # A saved/restored manifest takes precedence over regenerating splits.\n    df = assign_folds(df, settings[\'n_splits\'], settings[\'seed\'], saved if saved.exists() else supplied)\n    if saved.exists() and supplied:\n        requested = assign_folds(df.drop(columns=\'fold\'), settings[\'n_splits\'], settings[\'seed\'], supplied)\n        require(df.fold.equals(requested.fold), \'CNN folds differ from restored transformer folds\')\n    stable_columns = [\'file_name\', \'video_id\', \'label\', \'fold\']\n    protocol = {\'engine\': ENGINE_VERSION, \'task\': settings[\'task\'],\n                \'metadata_hash\': table_hash(df, [\'file_name\', \'video_id\', \'label\']),\n                \'fold_hash\': table_hash(df, stable_columns), \'archive_sha256\': archive_hash,\n                \'n_splits\': settings[\'n_splits\'], \'classes\': settings[\'source\'][\'classes\']}\n    protocol_path = task_root / \'data_protocol.json\'\n    if protocol_path.exists():\n        require(json.loads(protocol_path.read_text()) == protocol,\n                \'Dataset/folds changed in this output folder. Use a new RUN_NAME.\')\n    else:\n        save_json(protocol_path, protocol)\n    df.to_csv(saved, index=False)\n    save_json(task_root / \'data_sources.json\', {**settings[\'source\'],\n        \'metadata_file_sha256\': file_hash(metadata_path), \'archive_sha256\': archive_hash,\n        \'metadata_hash\': protocol[\'metadata_hash\'], \'fold_hash\': protocol[\'fold_hash\'],\n        \'seed\': settings[\'seed\'], \'splitter\': \'StratifiedGroupKFold\', \'shuffle\': True,\n        \'fold_source\': str(supplied) if supplied else \'stored manifest / seed 42 CNN-style generation\'})\n    audit = df.groupby([\'fold\', \'label\']).agg(\n        frames=(\'file_name\', \'size\'), videos_containing_class=(\'video_id\', \'nunique\')).reset_index()\n    audit.to_csv(task_root / \'fold_audit.csv\', index=False)\n    print(pd.crosstab(df.fold, df.label), flush=True)\n    print(\'Fold assignments saved:\', saved, flush=True)\n    return df, protocol\n\n\ndef normalize_model(value):\n    import re\n    key = re.sub(r\'[^a-z0-9]\', \'\', str(value).lower())\n    aliases = {}\n    for name, implementation in REGISTRY.items():\n        aliases[re.sub(r\'[^a-z0-9]\', \'\', name.lower())] = name\n        aliases[re.sub(r\'[^a-z0-9]\', \'\', implementation.lower())] = name\n    return aliases.get(key)\n\n\ndef select_models(defaults, ranking_csv=None, top_n=4, task=None):\n    if not ranking_csv:\n        names = [normalize_model(name) for name in defaults]\n        require(all(names) and len(names) == len(set(names)), \'Invalid/duplicate MODEL_NAMES\')\n        return names, {\'method\': \'editable candidate shortlist\',\n                       \'ranking_verified\': False, \'models\': names}\n    ranking = pd.read_csv(ranking_csv)\n    require(\'model\' in ranking, \'Ranking CSV needs a model column\')\n    if \'task\' in ranking and task:\n        normalized = ranking.task.astype(str).str.lower().str.replace(\' \', \'_\')\n        normalized = normalized.replace({\'poordilation\': \'poor_dilation\'})\n        ranking = ranking[normalized == task].copy()\n    metric = next((column for column in [\'val_macro_f1\', \'validation_macro_f1\']\n                   if column in ranking), None)\n    require(metric is not None, \'Use a previous validation macro-F1 summary, not test scores\')\n    ranking[\'canonical_model\'] = ranking.model.map(normalize_model)\n    ranking = ranking.dropna(subset=[\'canonical_model\']).copy()\n    def score(value):\n        value = str(value).strip()\n        return float(value[:-1]) / 100 if value.endswith(\'%\') else float(value)\n    ranking[\'selection_score\'] = ranking[metric].map(score)\n    require(ranking.selection_score.between(0, 1).all(), \'Validation scores must be fractions or % strings\')\n    require(not ranking.canonical_model.duplicated().any(), \'Ranking CSV needs one row per task/model\')\n    ranking = ranking.sort_values([\'selection_score\', \'canonical_model\'], ascending=[False, True])\n    require(len(ranking) >= top_n, f\'Ranking contains fewer than {top_n} supported models\')\n    selected = ranking.head(top_n)\n    names = selected.canonical_model.tolist()\n    return names, {\'method\': \'previous validation macro-F1\', \'ranking_verified\': True,\n        \'ranking_csv\': str(ranking_csv), \'ranking_sha256\': file_hash(ranking_csv),\n        \'metric\': metric, \'models\': names,\n        \'scores\': dict(zip(names, selected.selection_score.astype(float)))}\n\n\nclass FrameDataset(Dataset):\n    def __init__(self, dataframe, training):\n        self.df = dataframe.reset_index(drop=True)\n        ops = [transforms.Resize((224, 224))]\n        if training:\n            ops += [transforms.RandomHorizontalFlip(0.5), transforms.RandomVerticalFlip(0.3),\n                    transforms.RandomRotation(15), transforms.RandomAffine(0, translate=(0.1, 0.1))]\n        self.transform = transforms.Compose(ops + [transforms.ToTensor(),\n            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])\n\n    def __len__(self):\n        return len(self.df)\n\n    def __getitem__(self, index):\n        row = self.df.iloc[index]\n        with Image.open(row.image_path) as image:\n            tensor = self.transform(image.convert(\'RGB\'))\n        return tensor, int(row.label)\n\n\nclass TransformerClassifier(nn.Module):\n    def __init__(self, name, num_classes, pretrained=True):\n        super().__init__()\n        self.backbone = timm.create_model(REGISTRY[name], pretrained=pretrained, num_classes=0)\n        self.binary = num_classes == 2\n        self.classifier = nn.Linear(self.backbone.num_features, 1 if self.binary else num_classes)\n        # All backbone layers stay trainable, as in the supplied transformer notebooks.\n\n    def forward(self, images):\n        features = self.backbone(images)\n        require(isinstance(features, torch.Tensor) and features.ndim == 2,\n                \'Expected a pooled tensor from the timm backbone\')\n        features = torch.nn.functional.normalize(features, p=2, dim=1)\n        logits = self.classifier(features)\n        return logits.squeeze(-1) if self.binary else logits\n\n\ndef seed_all(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n\n\ndef seed_worker(worker_id):\n    seed = torch.initial_seed() % 2**32\n    np.random.seed(seed)\n    random.seed(seed)\n\n\ndef make_loaders(df, fold, cfg, device):\n    train_ds = FrameDataset(df[df.fold != fold], True)\n    valid_ds = FrameDataset(df[df.fold == fold], False)\n    train_generator = torch.Generator()\n    valid_generator = torch.Generator()\n    sample_generator = torch.Generator()\n    counts = train_ds.df.label.value_counts()\n    weights = train_ds.df.label.map(lambda value: 1 / counts[value]).to_numpy()\n    sampler = WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double),\n                                    len(train_ds), replacement=True, generator=sample_generator)\n    common = {\'batch_size\': cfg[\'batch_size\'], \'num_workers\': cfg[\'workers\'],\n              \'pin_memory\': device.type == \'cuda\', \'worker_init_fn\': seed_worker,\n              \'drop_last\': False}\n    # Workers restart each epoch; epoch-specific generator seeds make resume reproducible.\n    train = DataLoader(train_ds, sampler=sampler, generator=train_generator, **common)\n    valid = DataLoader(valid_ds, shuffle=False, generator=valid_generator, **common)\n    return train, valid, (train_generator, valid_generator, sample_generator)\n\n\ndef epoch_pass(model, loader, device, optimizer=None, scaler=None):\n    training = optimizer is not None\n    model.train(training)\n    criterion = nn.BCEWithLogitsLoss() if model.binary else nn.CrossEntropyLoss()\n    loss_sum, count, ys, ps = 0.0, 0, [], []\n    with torch.set_grad_enabled(training):\n        for images, labels in loader:\n            images = images.to(device, non_blocking=True)\n            labels = labels.to(device, non_blocking=True)\n            if training:\n                optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type, enabled=training and device.type == \'cuda\'):\n                logits = model(images)\n                loss = criterion(logits, labels.float() if model.binary else labels.long())\n            require(torch.isfinite(loss).item(), \'Non-finite loss\')\n            if training:\n                scaler.scale(loss).backward()\n                scaler.step(optimizer)\n                scaler.update()\n            loss_sum += loss.item() * len(labels)\n            count += len(labels)\n            if model.binary:\n                positive = logits.detach().float().sigmoid()\n                probabilities = torch.stack([1 - positive, positive], dim=1)\n            else:\n                probabilities = logits.detach().float().softmax(dim=1)\n            ys.append(labels.cpu().numpy())\n            ps.append(probabilities.cpu().numpy())\n    require(count > 0, \'Empty loader\')\n    return loss_sum / count, np.concatenate(ys), np.concatenate(ps)\n\n\ndef compute_metrics(y, probabilities, classes):\n    prediction = probabilities.argmax(axis=1)\n    labels = list(range(len(classes)))\n    precision, recall, f1, support = precision_recall_fscore_support(\n        y, prediction, labels=labels, zero_division=0)\n    metrics = {\'accuracy\': accuracy_score(y, prediction),\n               \'macro_f1\': f1_score(y, prediction, labels=labels, average=\'macro\', zero_division=0),\n               \'weighted_f1\': f1_score(y, prediction, labels=labels, average=\'weighted\', zero_division=0),\n               \'confusion_matrix\': confusion_matrix(y, prediction, labels=labels).tolist(),\n               \'n_frames\': len(y),\n               \'per_class\': {name: {\'precision\': precision[i], \'recall\': recall[i],\n                                   \'f1\': f1[i], \'support\': int(support[i])}\n                             for i, name in enumerate(classes)}}\n    try:\n        metrics[\'roc_auc\'] = roc_auc_score(y, probabilities[:, 1]) if len(classes) == 2 else \\\n            roc_auc_score(y, probabilities, labels=labels, multi_class=\'ovr\', average=\'macro\')\n    except ValueError:\n        metrics[\'roc_auc\'] = None\n    if len(classes) == 2:\n        metrics.update(poor_precision=precision[1], poor_recall=recall[1], specificity=recall[0])\n    return json_safe(metrics)\n\n\ndef save_checkpoint(path, value):\n    path = Path(path)\n    temp = path.with_suffix(\'.tmp\')\n    torch.save(value, temp)\n    temp.replace(path)\n\n\ndef rng_state():\n    return {\'python\': random.getstate(), \'numpy\': np.random.get_state(),\n            \'torch\': torch.get_rng_state(),\n            \'cuda\': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}\n\n\ndef restore_rng(state):\n    random.setstate(state[\'python\'])\n    np.random.set_state(state[\'numpy\'])\n    torch.set_rng_state(state[\'torch\'])\n    if torch.cuda.is_available() and state[\'cuda\']:\n        torch.cuda.set_rng_state_all(state[\'cuda\'])\n\n\ndef completed_fold(folder):\n    required = [\'complete.json\', \'protocol.json\', \'best.pth\', \'history.csv\',\n                \'validation_predictions.csv\', \'metrics.json\']\n    return all((Path(folder) / name).is_file() and (Path(folder) / name).stat().st_size\n               for name in required)\n\n\ndef train_fold(settings, name, fold, device_override=None):\n    cfg = settings[\'training\']\n    device = torch.device(device_override or \'cuda:0\')\n    if device.type == \'cuda\':\n        require(torch.cuda.is_available(), \'Enable Kaggle GPU T4 x2\')\n    df = pd.read_csv(settings[\'manifest\'])\n    validate_folds(df, settings[\'n_splits\'])\n    require(table_hash(df, [\'file_name\', \'video_id\', \'label\', \'fold\']) ==\n            settings[\'data_protocol\'][\'fold_hash\'], \'Worker manifest changed after data preparation\')\n    fold_seed = int(settings[\'seed\']) + int(fold)\n    seed_all(fold_seed)\n    classes = settings[\'source\'][\'classes\']\n    folder = Path(settings[\'task_root\']) / \'models\' / model_slug(name) / f\'fold_{fold}\'\n    folder.mkdir(parents=True, exist_ok=True)\n    protocol = {\'engine_version\': ENGINE_VERSION, \'engine_sha256\': settings[\'engine_sha256\'],\n                \'task\': settings[\'task\'], \'model\': name, \'timm_name\': REGISTRY[name],\n                \'classes\': classes, \'fold\': fold, \'fold_seed\': fold_seed,\n                \'n_splits\': settings[\'n_splits\'], \'training\': cfg,\n                \'data_protocol\': settings[\'data_protocol\'],\n                \'checkpoint_monitor\': \'validation_loss\',\n                \'versions\': {\'torch\': str(torch.__version__), \'timm\': timm.__version__,\n                             \'sklearn\': sklearn.__version__, \'numpy\': np.__version__,\n                             \'python\': platform.python_version()}}\n    protocol_path = folder / \'protocol.json\'\n    if protocol_path.exists():\n        require(json.loads(protocol_path.read_text()) == protocol,\n                f\'Run settings/code/version changed for {name}, fold {fold}. Use a new RUN_NAME.\')\n    else:\n        save_json(protocol_path, protocol)\n    last, best_path = folder / \'last.pth\', folder / \'best.pth\'\n    if completed_fold(folder):\n        if last.exists() and not cfg[\'keep_last_for_completed\']:\n            last.unlink()\n        print(\'Skipping completed fold:\', name, fold, flush=True)\n        return\n    model = TransformerClassifier(name, len(classes), pretrained=cfg[\'pretrained\'] and not last.exists())\n    model = model.to(device)\n    save_json(folder / \'pretrained_config.json\', getattr(model.backbone, \'pretrained_cfg\', {}))\n    save_json(folder / \'parameter_counts.json\', {\n        \'total\': sum(p.numel() for p in model.parameters()),\n        \'trainable\': sum(p.numel() for p in model.parameters() if p.requires_grad)})\n    optimizer = torch.optim.AdamW([\n        {\'params\': model.backbone.parameters(), \'lr\': cfg[\'backbone_lr\'], \'name\': \'backbone\'},\n        {\'params\': model.classifier.parameters(), \'lr\': cfg[\'head_lr\'], \'name\': \'head\'},\n    ], weight_decay=cfg[\'weight_decay\'])\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg[\'epochs\'])\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=device.type == \'cuda\')\n    train_loader, valid_loader, generators = make_loaders(df, fold, cfg, device)\n    start, best, bad, history = 0, float(\'inf\'), 0, []\n    if last.exists():\n        state = torch.load(last, map_location=\'cpu\', weights_only=False)\n        require(state[\'protocol\'] == protocol, \'Resume checkpoint protocol mismatch\')\n        require(best_path.exists(), \'Partial fold is missing its best checkpoint\')\n        model.load_state_dict(state[\'model\'])\n        optimizer.load_state_dict(state[\'optimizer\'])\n        scheduler.load_state_dict(state[\'scheduler\'])\n        scaler.load_state_dict(state[\'scaler\'])\n        start, best, bad, history = state[\'epoch\'] + 1, state[\'best_loss\'], state[\'bad_epochs\'], state[\'history\']\n        restore_rng(state[\'rng\'])\n        del state\n        print(f\'Resuming {name}, fold {fold}, from epoch {start + 1}\', flush=True)\n    for epoch in range(start, cfg[\'epochs\']):\n        if bad >= cfg[\'patience\']:\n            break\n        epoch_started = time.perf_counter()\n        for offset, generator in enumerate(generators):\n            generator.manual_seed(fold_seed + epoch * 10000 + offset)\n        lrs = {group[\'name\'] + \'_lr\': float(group[\'lr\']) for group in optimizer.param_groups}\n        train_loss, yt, pt = epoch_pass(model, train_loader, device, optimizer, scaler)\n        val_loss, yv, pv = epoch_pass(model, valid_loader, device)\n        val_metrics = compute_metrics(yv, pv, classes)\n        row = {\'epoch\': epoch + 1, \'train_loss\': train_loss, \'val_loss\': val_loss,\n               \'train_accuracy\': float(accuracy_score(yt, pt.argmax(1))),\n               \'val_accuracy\': val_metrics[\'accuracy\'], \'val_macro_f1\': val_metrics[\'macro_f1\'],\n               \'epoch_seconds\': time.perf_counter() - epoch_started, **lrs}\n        history.append(row)\n        weights = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}\n        if val_loss < best:\n            best, bad = val_loss, 0\n            save_checkpoint(best_path, {\'model\': weights, \'epoch\': epoch + 1,\n                                       \'val_loss\': val_loss, \'protocol\': protocol})\n        else:\n            bad += 1\n        scheduler.step()\n        pd.DataFrame(history).to_csv(folder / \'history.csv\', index=False)\n        save_checkpoint(last, {\'model\': weights, \'optimizer\': optimizer.state_dict(),\n            \'scheduler\': scheduler.state_dict(), \'scaler\': scaler.state_dict(),\n            \'epoch\': epoch, \'best_loss\': best, \'bad_epochs\': bad, \'history\': history,\n            \'rng\': rng_state(), \'protocol\': protocol})\n        print(settings[\'task\'], name, f\'fold={fold}\', row, flush=True)\n        del weights\n    checkpoint = torch.load(best_path, map_location=\'cpu\', weights_only=False)\n    model.load_state_dict(checkpoint[\'model\'])\n    val_loss, y, probabilities = epoch_pass(model, valid_loader, device)\n    predictions = valid_loader.dataset.df[[\'row_id\', \'file_name\', \'video_id\', \'label\', \'fold\']].copy()\n    require(np.array_equal(predictions.label.to_numpy(), y), \'Prediction order mismatch\')\n    predictions[\'prediction\'] = probabilities.argmax(1)\n    for index in range(len(classes)):\n        predictions[f\'prob_{index}\'] = probabilities[:, index]\n    predictions.to_csv(folder / \'validation_predictions.csv\', index=False)\n    result = {\'task\': settings[\'task\'], \'model\': name, \'fold\': fold,\n              \'best_epoch\': checkpoint[\'epoch\'], \'epochs_trained\': len(history),\n              \'best_val_loss\': checkpoint[\'val_loss\'], \'selected_checkpoint_val_loss\': val_loss,\n              \'train_frames\': len(train_loader.dataset), \'val_frames\': len(valid_loader.dataset),\n              \'train_groups\': train_loader.dataset.df.video_id.nunique(),\n              \'val_groups\': valid_loader.dataset.df.video_id.nunique(),\n              \'metrics\': compute_metrics(y, probabilities, classes)}\n    save_json(folder / \'metrics.json\', result)\n    save_json(folder / \'complete.json\', {\'best_epoch\': checkpoint[\'epoch\'],\n                                        \'fold_hash\': settings[\'data_protocol\'][\'fold_hash\']})\n    # Best weights stay; only finished-fold resume states are removed to save Kaggle disk.\n    if not cfg[\'keep_last_for_completed\'] and last.exists():\n        last.unlink()\n    print(\'COMPLETED:\', settings[\'task\'], name, fold, result[\'metrics\'], flush=True)\n    del model, optimizer, scheduler, scaler, checkpoint, train_loader, valid_loader\n    gc.collect()\n    if device.type == \'cuda\':\n        torch.cuda.empty_cache()\n\n\ndef save_plot(matrix, classes, path):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    fig, ax = plt.subplots(figsize=(6.5, 5.5))\n    im = ax.imshow(matrix, cmap=\'Blues\')\n    for i in range(len(classes)):\n        for j in range(len(classes)):\n            ax.text(j, i, int(matrix[i][j]), ha=\'center\', va=\'center\',\n                    color=\'white\' if matrix[i][j] > np.max(matrix) / 2 else \'black\')\n    ax.set(xticks=range(len(classes)), yticks=range(len(classes)), xticklabels=classes,\n           yticklabels=classes, xlabel=\'Predicted class\', ylabel=\'Reference class\')\n    plt.setp(ax.get_xticklabels(), rotation=25, ha=\'right\')\n    fig.colorbar(im, ax=ax)\n    fig.tight_layout()\n    fig.savefig(path, dpi=150)\n    plt.close(fig)\n\n\ndef summarize(settings, models, plots=True):\n    root = Path(settings[\'task_root\'])\n    root.mkdir(parents=True, exist_ok=True)\n    rows, statuses = [], []\n    classes = settings[\'source\'][\'classes\']\n    metric_names = [\'accuracy\', \'macro_f1\', \'weighted_f1\', \'roc_auc\']\n    if len(classes) == 2:\n        metric_names += [\'poor_precision\', \'poor_recall\', \'specificity\']\n    for name in models:\n        directory = root / \'models\' / model_slug(name)\n        directory.mkdir(parents=True, exist_ok=True)\n        fold_rows, frames = [], []\n        for fold in range(settings[\'n_splits\']):\n            folder = directory / f\'fold_{fold}\'\n            done = completed_fold(folder)\n            statuses.append({\'model\': name, \'fold\': fold, \'status\': \'complete\' if done else\n                             \'partial\' if (folder / \'last.pth\').exists() else \'not_started\'})\n            if not done:\n                continue\n            result = json.loads((folder / \'metrics.json\').read_text())\n            require(result[\'model\'] == name and result[\'fold\'] == fold and result[\'task\'] == settings[\'task\'],\n                    \'Wrong model/task/fold in saved metrics\')\n            fold_rows.append({\'fold\': fold, \'best_epoch\': result[\'best_epoch\'],\n                \'epochs_trained\': result[\'epochs_trained\'], \'best_val_loss\': result[\'best_val_loss\'],\n                **{key: result[\'metrics\'].get(key) for key in metric_names}})\n            frames.append(pd.read_csv(folder / \'validation_predictions.csv\'))\n            if plots and not (folder / \'confusion_matrix.png\').exists():\n                save_plot(np.asarray(result[\'metrics\'][\'confusion_matrix\']), classes,\n                          folder / \'confusion_matrix.png\')\n            if plots and not (folder / \'training_curves.png\').exists():\n                import matplotlib.pyplot as plt\n                history = pd.read_csv(folder / \'history.csv\')\n                fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))\n                axes[0].plot(history.epoch, history.train_loss, label=\'Train\')\n                axes[0].plot(history.epoch, history.val_loss, label=\'Validation\')\n                axes[0].set(xlabel=\'Epoch\', ylabel=\'Loss\')\n                axes[0].legend()\n                axes[1].plot(history.epoch, history.val_macro_f1)\n                axes[1].set(xlabel=\'Epoch\', ylabel=\'Validation macro-F1\', ylim=(0, 1))\n                fig.tight_layout()\n                fig.savefig(folder / \'training_curves.png\', dpi=150)\n                plt.close(fig)\n        if not fold_rows:\n            continue\n        fold_metrics = pd.DataFrame(fold_rows).sort_values(\'fold\')\n        fold_metrics.to_csv(directory / \'fold_metrics.csv\', index=False)\n        oof = pd.concat(frames, ignore_index=True).sort_values(\'row_id\')\n        require(not oof.row_id.duplicated().any(), \'A frame occurs in multiple OOF predictions\')\n        oof.to_csv(directory / \'out_of_fold_predictions.csv\', index=False)\n        probabilities = oof[[f\'prob_{i}\' for i in range(len(classes))]].to_numpy()\n        pooled = compute_metrics(oof.label.to_numpy(), probabilities, classes)\n        complete = len(fold_rows) == settings[\'n_splits\']\n        if complete:\n            expected = pd.read_csv(settings[\'manifest\'])\n            require(set(oof.row_id) == set(expected.row_id), \'OOF coverage differs from the dataset\')\n        save_json(directory / \'oof_metrics.json\', {\'complete\': complete,\n            \'completed_folds\': len(fold_rows), \'metrics\': pooled})\n        pd.DataFrame(pooled[\'confusion_matrix\'], index=classes, columns=classes).to_csv(\n            directory / \'oof_confusion_matrix.csv\')\n        pd.DataFrame(pooled[\'per_class\']).T.to_csv(directory / \'per_class_metrics.csv\')\n        if plots:\n            save_plot(np.asarray(pooled[\'confusion_matrix\']), classes, directory / \'oof_confusion_matrix.png\')\n        summary = {\'model\': name, \'completed_folds\': len(fold_rows),\n                   \'expected_folds\': settings[\'n_splits\'], \'complete\': complete,\n                   \'oof_frames\': len(oof)}\n        for metric in metric_names:\n            values = pd.to_numeric(fold_metrics[metric], errors=\'coerce\')\n            summary[metric + \'_mean\'] = values.mean()\n            summary[metric + \'_std\'] = values.std(ddof=1) if len(values) > 1 else None\n            summary[\'oof_\' + metric] = pooled.get(metric)\n        rows.append(summary)\n    status = pd.DataFrame(statuses)\n    status.to_csv(root / \'run_status.csv\', index=False)\n    summary = pd.DataFrame(rows)\n    if not summary.empty:\n        summary = summary.sort_values([\'complete\', \'macro_f1_mean\'], ascending=[False, False])\n    summary.to_csv(root / \'model_summary.csv\', index=False)\n    return summary, status\n\n\ndef restore_outputs(settings, url=None, local_zip=None):\n    if not url and not local_zip:\n        return\n    if local_zip:\n        path = Path(local_zip)\n        require(path.is_file(), f\'Missing resume archive: {path}\')\n    else:\n        import gdown\n        tag = hashlib.sha256(url.encode()).hexdigest()[:12]\n        path = Path(settings[\'root\']) / (settings[\'run_name\'] + \'_resume_\' + tag + \'.zip\')\n        if not path.exists():\n            result = gdown.download(url=url, output=str(path), fuzzy=True, quiet=False)\n            require(result is not None, \'Resume archive download failed\')\n    parent = Path(settings[\'task_root\']).parents[1]\n    require(zipfile.is_zipfile(path), \'Resume file is not a ZIP archive\')\n    with zipfile.ZipFile(path) as archive:\n        validate_zip_members(archive, parent)\n        members = [info for info in archive.infolist() if not info.is_dir()]\n        require(members and all(Path(info.filename).parts[0] == settings[\'run_name\'] for info in members),\n                \'Resume ZIP must be exported by this task/RUN_NAME notebook\')\n        # Never replace a newer, different checkpoint with an older restore archive.\n        for info in members:\n            destination = parent / info.filename\n            if destination.exists():\n                require(hashlib.sha256(archive.read(info)).hexdigest() == file_hash(destination),\n                        f\'Resume conflicts with an existing output: {destination}\')\n        archive.extractall(parent)\n    print(\'Restored previous transformer outputs:\', path, flush=True)\n\n\ndef launch(settings, models):\n    import fcntl\n    require(torch.cuda.is_available(), \'Enable a Kaggle GPU accelerator before the training cell\')\n    count = min(2, torch.cuda.device_count())\n    root = Path(settings[\'task_root\'])\n    root.mkdir(parents=True, exist_ok=True)\n    lock = (root / \'launcher.lock\').open(\'w\')\n    try:\n        fcntl.flock(lock.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)\n    except BlockingIOError:\n        lock.close()\n        raise RuntimeError(\'Another launcher is active for this output folder\')\n    settings_path = root / \'worker_settings.json\'\n    save_json(settings_path, settings)\n    logs = root / \'logs\'\n    logs.mkdir(exist_ok=True)\n    pending = [(name, fold) for fold in range(settings[\'n_splits\']) for name in models]\n    print(f\'{len(models)} models x {settings["n_splits"]} folds = {len(pending)} jobs; \'\n          f\'{count} GPU worker(s). Completed jobs are checked and skipped.\', flush=True)\n    visible = os.environ.get(\'CUDA_VISIBLE_DEVICES\')\n    tokens = visible.split(\',\') if visible else [str(i) for i in range(torch.cuda.device_count())]\n    require(len(tokens) >= count, \'CUDA device mapping is inconsistent\')\n    active, failed = {}, []\n    started = last_report = time.monotonic()\n    try:\n        while pending or active:\n            for gpu in range(count):\n                if gpu in active or not pending:\n                    continue\n                name, fold = pending.pop(0)\n                log_path = logs / f\'{model_slug(name)}_fold_{fold}.log\'\n                log = log_path.open(\'a\', buffering=1)\n                env = os.environ.copy()\n                env.update(CUDA_VISIBLE_DEVICES=tokens[gpu].strip(),\n                    OMP_NUM_THREADS=\'1\', MKL_NUM_THREADS=\'1\', OPENBLAS_NUM_THREADS=\'1\',\n                    PYTHONHASHSEED=str(settings[\'seed\'] + fold), PYTHONUNBUFFERED=\'1\')\n                try:\n                    process = subprocess.Popen([sys.executable, settings[\'engine_path\'],\n                        \'--settings\', str(settings_path), \'--model\', name, \'--fold\', str(fold)],\n                        stdout=log, stderr=subprocess.STDOUT, env=env, start_new_session=True)\n                except BaseException:\n                    log.close()\n                    raise\n                active[gpu] = (process, name, fold, log, log_path)\n                print(f\'GPU {gpu}: {name}, fold {fold + 1}/{settings["n_splits"]} started; {log_path}\', flush=True)\n            for gpu, (process, name, fold, log, log_path) in list(active.items()):\n                code = process.poll()\n                if code is None:\n                    continue\n                log.close()\n                del active[gpu]\n                print(f\'GPU {gpu}: {name}, fold {fold + 1} finished; exit={code}\', flush=True)\n                if code:\n                    failed.append((name, fold))\n                    print(log_path.read_text(errors=\'replace\')[-5000:], flush=True)\n                summarize(settings, models)\n            if time.monotonic() - last_report >= 30:\n                print(f\'Elapsed {(time.monotonic() - started) / 60:.1f} min; \'\n                      f\'{len(pending)} queued, {len(active)} running.\', flush=True)\n                for _, name, fold, _, log_path in active.values():\n                    tail = log_path.read_text(errors=\'replace\').splitlines()[-1:]\n                    if tail:\n                        print(name, \'fold\', fold + 1, tail[0][-400:], flush=True)\n                last_report = time.monotonic()\n            if active:\n                time.sleep(2)\n    finally:\n        for process, _, _, _, _ in active.values():\n            if process.poll() is None:\n                try:\n                    os.killpg(process.pid, signal.SIGTERM)\n                except ProcessLookupError:\n                    pass\n        for process, _, _, log, _ in active.values():\n            try:\n                process.wait(timeout=10)\n            except subprocess.TimeoutExpired:\n                try:\n                    os.killpg(process.pid, signal.SIGKILL)\n                except ProcessLookupError:\n                    pass\n                process.wait()\n            log.close()\n        try:\n            summarize(settings, models)\n        finally:\n            lock.close()\n    require(not failed, f\'Some jobs failed: {failed}. Inspect their logs, then rerun to resume.\')\n    return summarize(settings, models)\n\n\ndef export_outputs(settings):\n    run_root = Path(settings[\'task_root\']).parent\n    archive_path = Path(settings[\'root\']) / (settings[\'run_name\'] + \'.zip\')\n    with zipfile.ZipFile(archive_path, \'w\', compression=zipfile.ZIP_STORED, allowZip64=True) as archive:\n        for path in sorted(run_root.rglob(\'*\')):\n            if path.is_file() and path.suffix != \'.tmp\' and path.name != \'launcher.lock\':\n                archive.write(path, path.relative_to(run_root.parent))\n    return archive_path\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--settings\', required=True)\n    parser.add_argument(\'--model\', required=True, choices=list(REGISTRY))\n    parser.add_argument(\'--fold\', required=True, type=int)\n    args = parser.parse_args()\n    settings = json.loads(Path(args.settings).read_text())\n    require(torch.cuda.is_available() and torch.cuda.device_count() == 1,\n            \'A worker must see exactly one assigned GPU\')\n    torch.set_num_threads(1)\n    torch.set_num_interop_threads(1)\n    train_fold(settings, args.model, args.fold)\n'
ENGINE_PATH = PROJECT_DIR / 'transformer_kfold.py'
ENGINE_PATH.write_text(ENGINE_SOURCE, encoding='utf-8')
spec = importlib.util.spec_from_file_location('transformer_kfold_' + TASK, ENGINE_PATH)
engine = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = engine
spec.loader.exec_module(engine)
SETTINGS = dict(
    root=str(ROOT.resolve()), task=TASK, run_name=RUN_NAME,
    task_root=str(TASK_ROOT.resolve()), data_root=str(DATA_ROOT.resolve()),
    source=SOURCE, local_files=LOCAL_FILES, n_splits=N_SPLITS, seed=SEED,
    cnn_folds_csv=CNN_FOLDS_CSV, validate_images=VALIDATE_IMAGES,
    training=TRAINING, manifest=str((TASK_ROOT / 'fold_assignments.csv').resolve()),
    engine_path=str(ENGINE_PATH.resolve()), engine_sha256=engine.file_hash(ENGINE_PATH),
)
print('Standalone worker project:', ENGINE_PATH)


## 5. Restore previous K-fold outputs, if any
Leave the resume fields empty on the first run. A restored output ZIP must have this notebook's `RUN_NAME` as its top-level folder. Different existing files are not overwritten.


In [ ]:
engine.restore_outputs(SETTINGS, RESUME_OUTPUTS_ZIP_URL, RESUME_LOCAL_ZIP)
saved_engine = TASK_ROOT / 'engine_source.py'
if saved_engine.exists() and engine.file_hash(saved_engine) != SETTINGS['engine_sha256']:
    raise ValueError('Engine code changed from the saved run. Use a new RUN_NAME.')
shutil.copy2(ENGINE_PATH, saved_engine)


## 6. Download and verify original data; create or reuse five folds
Expected: **2,541 frame records, 645 video groups**. Original one-hot labels are converted to zero-based model indices. Poor-dilation exact duplicate records are removed just as in the CNN K-fold notebook; mixed-label videos remain together. Dataset labels and identifiers must match the audited original checksum.

The saved manifest preserves original filenames and video IDs. No mapped-name exclusion or renaming is applied. Data checks finish before model training starts.


In [ ]:
DATA, DATA_PROTOCOL = engine.prepare_task(SETTINGS)
SETTINGS['data_protocol'] = DATA_PROTOCOL
from IPython.display import display
display(DATA.groupby(['fold', 'label']).agg(
    frames=('file_name', 'size'), videos_containing_class=('video_id', 'nunique')))
print('Every video belongs to exactly one fold; all folds contain all task classes.')
print('Class order:', SOURCE['classes'])


## 7. Select the four models
When a prior validation summary is provided, this cell chooses its top four supported transformer models. Otherwise it prints the editable candidate list. The selection method is saved alongside the results; no performance ranking is invented.


In [ ]:
SELECTED_MODELS, SELECTION = engine.select_models(
    MODEL_NAMES, PRIOR_RESULTS_CSV, TOP_N, TASK)
engine.save_json(TASK_ROOT / 'model_selection.json', SELECTION)
engine.save_json(TASK_ROOT / 'notebook_settings.json', SETTINGS)
print('Selected models:', SELECTED_MODELS)
print('Selection method:', SELECTION['method'])
print('Jobs:', len(SELECTED_MODELS) * N_SPLITS)
if not SELECTION['ranking_verified']:
    print('No prior scores supplied: these are candidates, not verified top performers.')


## 8. Train / resume on both GPUs
This is the long-running cell. It saves a separate console log for each model/fold and prints active-job progress every 30 seconds. Best weights are selected by the held-out fold's validation loss. Full backbone fine-tuning uses LR 1e-5, the new head uses LR 1e-4, with cosine decay, mixed precision, and early stopping.

Interrupting a session retains the last fully saved epoch of each unfinished fold. Rerun the notebook with the same settings to continue. Completed folds are checked and skipped automatically.


In [ ]:
if RUN_TRAINING:
    SUMMARY, STATUS = engine.launch(SETTINGS, SELECTED_MODELS)
else:
    SUMMARY, STATUS = engine.summarize(SETTINGS, SELECTED_MODELS)
    print('Data audit only. Set RUN_TRAINING=True in Section 2 to train.')
display(STATUS)
display(SUMMARY)


## 9. Results and completion check
Results are frame-level. A group's mixed poor-dilation labels are not replaced by an arbitrary video-level label.

- Per fold: `best.pth`, `history.csv`, `protocol.json`, `metrics.json`, `validation_predictions.csv`, confusion matrix, and training curves.
- Per model: fold metrics, out-of-fold predictions, pooled metrics, per-class metrics, and confusion matrix.
- Task: `fold_assignments.csv`, class/fold audit, selected-model record, environment/config records, `run_status.csv`, and `model_summary.csv`.

`*_mean` and `*_std` summarize completed folds; `oof_*` metrics pool the out-of-fold frames. Partial models remain explicitly marked. A complete five-fold model predicts every original frame exactly once in the OOF file. Scores are fractions in [0,1].


In [ ]:
SUMMARY, STATUS = engine.summarize(SETTINGS, SELECTED_MODELS)
display(STATUS)
display(SUMMARY)
finished = int((STATUS.status == 'complete').sum())
expected = len(SELECTED_MODELS) * N_SPLITS
print(f'Completed fold jobs: {finished}/{expected}')
print('All planned folds complete:', finished == expected)
print('Download this complete task folder:', TASK_ROOT)


## 10. Export outputs for download or the next Kaggle session
Set `EXPORT_ZIP=True` after a complete **or partial** session. The ZIP includes best weights, results, and resume states of unfinished folds. Upload it to your own Drive and paste its view link into Section 2 in the next session. Creating a ZIP requires extra working-disk space.


In [ ]:
from IPython.display import FileLink
EXPORT_ZIP = False
if EXPORT_ZIP:
    OUTPUT_ZIP = engine.export_outputs(SETTINGS)
    print('Exported:', OUTPUT_ZIP)
    display(FileLink(str(OUTPUT_ZIP.relative_to(ROOT))))
else:
    print('Set EXPORT_ZIP=True to package complete or partial outputs.')
